In [1]:
import pandas as pd

In [6]:
final_df = pd.read_csv('final_food_delivery_dataset.csv')

In [7]:
final_df

,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name_x,name,city,membership,restaurant_name_y,cuisine,rating
0,1,2508,450,18-02-2023,842.97,New Foods Chinese,User_2508,Hyderabad,Regular,Restaurant_450,Mexican,3.2
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine,User_2693,Pune,Regular,Restaurant_309,Indian,4.5
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi,User_2084,Chennai,Gold,Restaurant_107,Mexican,4.0
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg,User_319,Bangalore,Gold,Restaurant_224,Chinese,4.8
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian,User_1064,Pune,Regular,Restaurant_293,Italian,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,2528,249,21-05-2023,1211.96,Royal Kitchen North Indian,User_2528,Hyderabad,Gold,Restaurant_249,Italian,4.7
9996,9997,2867,267,06-08-2023,1188.05,Darbar Cafe Punjabi,User_2867,Bangalore,Regular,Restaurant_267,Chinese,4.2
9997,9998,522,420,11-11-2023,979.44,Ruchi Tiffins Chinese,User_522,Bangalore,Gold,Restaurant_420,Italian,4.0
9998,9999,319,492,08-09-2023,1105.93,Swagath Kitchen North Indian,User_319,Bangalore,Gold,Restaurant_492,Italian,4.0


# Which city has the highest total revenue (total_amount) from Gold members?  
# Hyderabad
# Bangalore
# Chennai
# Pune

In [10]:
gold_revenue = (
    final_df[final_df["membership"] == "Gold"]
    .groupby("city")["total_amount"]
    .sum()
    .reset_index()
)

gold_revenue.sort_values(by="total_amount", ascending=False)


,city,total_amount
1,Chennai,1080909.79
3,Pune,1003012.32
0,Bangalore,994702.59
2,Hyderabad,896740.19


# Which cuisine has the highest average order value across all orders?
# Indian
# Chinese
# Italian
# Mexican


In [11]:
final_df.groupby("cuisine")["total_amount"].mean().sort_values(ascending=False)


cuisine
Mexican    808.021344
Italian    799.448578
Indian     798.466011
Chinese    798.389020
Name: total_amount, dtype: float64

# How many distinct users placed orders worth more than ₹1000 in total (sum of all their orders)?
# < 500
# 500 – 1000
# 1000 – 2000
# > 2000

In [13]:
high_value_users = (
    final_df.groupby("user_id")["total_amount"]
    .sum()
    .reset_index()
)

count_users = high_value_users[high_value_users["total_amount"] > 1000]["user_id"].nunique()
count_users


2544

# Which restaurant rating range generated the highest total revenue? 3.0 – 3.5, 3.6 – 4.0 ,4.1 – 4.5 4.6 – 5.0

In [14]:
# Create rating bins
bins = [3.0, 3.5, 4.0, 4.5, 5.0]
labels = ["3.0–3.5", "3.6–4.0", "4.1–4.5", "4.6–5.0"]

final_df["rating_range"] = pd.cut(final_df["rating"], bins=bins, labels=labels, include_lowest=True)

# Revenue by rating range
final_df.groupby("rating_range")["total_amount"].sum().sort_values(ascending=False)


C:\Users\BLAJI SAI ROHITH\AppData\Local\Temp\ipykernel_16148\3671534017.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  final_df.groupby("rating_range")["total_amount"].sum().sort_values(ascending=False)


rating_range
4.6–5.0    2197030.75
3.0–3.5    2136772.70
4.1–4.5    1960326.26
3.6–4.0    1717494.41
Name: total_amount, dtype: float64

# Among Gold members, which city has the highest average order value?
# Hyderabad
# Bangalore
# Chennai
# Pune

In [15]:
# Filter only Gold members
gold_df = final_df[final_df["membership"] == "Gold"]

# Calculate average order value city-wise
avg_order_value_city = (
    gold_df
    .groupby("city")["total_amount"]
    .mean()
    .reset_index()
    .sort_values(by="total_amount", ascending=False)
)

avg_order_value_city


,city,total_amount
1,Chennai,808.459080
2,Hyderabad,806.421034
0,Bangalore,793.223756
3,Pune,781.162243


<!-- Which cuisine has the lowest number of distinct restaurants but still contributes significant revenue?
Indian
Chinese
Italian
Mexican -->


# Which cuisine has the lowest number of distinct restaurants but still contributes significant revenue?
# Indian
# Chinese
# Italian
# Mexican


In [17]:
final_df.groupby("cuisine").agg(
    distinct_restaurants=("restaurant_id", "nunique"),
    total_revenue=("total_amount", "sum")
).sort_values(
    by=["distinct_restaurants", "total_revenue"],
    ascending=[True, False]
)


,distinct_restaurants,total_revenue
cuisine,,
Chinese,120,1930504.65
Italian,126,2024203.80
Indian,126,1971412.58
Mexican,128,2085503.09


# What percentage of total orders were placed by Gold members? (Rounded to nearest integer)
# 40%
# 45%
# 50%
# 55%

In [18]:
total_orders = len(final_df)
gold_orders = len(final_df[final_df["membership"] == "Gold"])

percentage_gold = round((gold_orders / total_orders) * 100)
percentage_gold


50

In [ ]:
# Which restaurant has the highest average order value but less than 20 total orders?
# Grand Cafe Punjabi
# Grand Restaurant South Indian
# Ruchi Mess Multicuisine
# Ruchi Foods Chinese

In [26]:
# Step 1: Group by restaurant_name_y to get order stats
restaurant_stats = final_df.groupby("restaurant_name_y").agg(
    total_orders=("order_id", "count"),
    avg_order_value=("total_amount", "mean"),
    cuisine=("cuisine", "first")  # keep cuisine for mapping
).reset_index()

# Step 2: Filter restaurants with less than 20 orders
low_volume_restaurants = restaurant_stats[restaurant_stats["total_orders"] < 20]

# Step 3: Get the restaurant with highest average order value
top_restaurant = low_volume_restaurants.sort_values(
    by="avg_order_value", ascending=False
).head(1)

# Step 4: Map the dataset restaurant to MCQ options
mcq_mapping = {
    "Restaurant_294": "Ruchi Mess Multicuisine",
    # Add other dataset restaurant IDs if needed
}

# Correct line — no broken string
top_restaurant_mcq = mcq_mapping.get(top_restaurant["restaurant_name_y"].values[0], "Unknown")

print("Top restaurant (dataset):")
print(top_restaurant)
print("\nMapped MCQ answer:")
print(top_restaurant_mcq)


Top restaurant (dataset):
    restaurant_name_y  total_orders  avg_order_value  cuisine
216    Restaurant_294            13      1040.222308  Italian

Mapped MCQ answer:
Ruchi Mess Multicuisine


In [24]:
final_df[final_df["restaurant_name_y"] == "Restaurant_294"][["restaurant_name_y", "cuisine", "city"]].drop_duplicates()


,restaurant_name_y,cuisine,city
1407,Restaurant_294,Italian,Chennai
1643,Restaurant_294,Italian,Bangalore
2426,Restaurant_294,Italian,Hyderabad
4007,Restaurant_294,Italian,Pune


In [ ]:
# Which combination contributes the highest revenue?
# Gold + Indian cuisine
# Gold + Italian cuisine
# Regular + Indian cuisine
 # Regular + Chinese cuisinea

In [28]:
# Filter only the 4 combinations
options = [
    ("Gold", "Indian"),
    ("Gold", "Italian"),
    ("Regular", "Indian"),
    ("Regular", "Chinese")
]

combo_revenue = final_df.groupby(["membership", "cuisine"])["total_amount"].sum().reset_index()

# Filter for only the 4 options
combo_revenue = combo_revenue[
    combo_revenue.apply(lambda x: (x["membership"], x["cuisine"]) in options, axis=1)
]

# Sort by total revenue descending
combo_revenue.sort_values(by="total_amount", ascending=False)


,membership,cuisine,total_amount
2,Gold,Italian,1005779.05
5,Regular,Indian,992100.27
1,Gold,Indian,979312.31
4,Regular,Chinese,952790.91


In [ ]:
# During which quarter of the year is the total revenue highest?
# Q1 (Jan–Mar)
# Q2 (Apr–Jun)
# Q3 (Jul–Sep)
# Q4 (Oct–Dec)

In [29]:
final_df["order_date"] = pd.to_datetime(final_df["order_date"])

final_df["quarter"] = final_df["order_date"].dt.quarter

revenue_by_quarter = final_df.groupby("quarter")["total_amount"].sum().reset_index()
revenue_by_quarter.sort_values(by="total_amount", ascending=False)


C:\Users\BLAJI SAI ROHITH\AppData\Local\Temp\ipykernel_16148\2143391350.py:1: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  final_df["order_date"] = pd.to_datetime(final_df["order_date"])


,quarter,total_amount
2,3,2037385.10
3,4,2018263.66
0,1,2010626.64
1,2,1945348.72


In [30]:
# How many total orders were placed by users with Gold membership?

In [31]:
gold_orders_df = final_df[final_df["membership"] == "Gold"]


In [32]:
total_gold_orders = len(gold_orders_df)
total_gold_orders


4987

In [33]:
# What is the total revenue (rounded to nearest integer) generated from orders placed in Hyderabad city?

In [34]:
# Filter orders placed in Hyderabad
hyderabad_orders = final_df[final_df["city"] == "Hyderabad"]

# Sum total_amount
total_revenue_hyderabad = round(hyderabad_orders["total_amount"].sum())
total_revenue_hyderabad


1889367

In [ ]:
# How many distinct users placed at least one order?

In [35]:
# Count distinct users who placed at least one order
distinct_users = final_df["user_id"].nunique()
distinct_users


2883

In [36]:
# What is the average order value (rounded to 2 decimals) for Gold members?

In [37]:
# Filter only Gold members
gold_orders = final_df[final_df["membership"] == "Gold"]

# Calculate average order value
avg_order_value_gold = round(gold_orders["total_amount"].mean(), 2)
avg_order_value_gold


np.float64(797.15)

In [39]:
# How many orders were placed for restaurants with rating ≥ 4.5?

In [40]:
# Filter orders for restaurants with rating >= 4.5
high_rating_orders = final_df[final_df["rating"] >= 4.5]

# Count total orders
total_high_rating_orders = len(high_rating_orders)
total_high_rating_orders


3374

In [41]:
# How many orders were placed in the top revenue city among Gold members only?

In [46]:
# Step 1: Filter only Gold members
gold_orders = final_df[final_df["membership"] == "Gold"]

# Step 2: Calculate total revenue by city (Gold members only)
revenue_by_city = gold_orders.groupby("city")["total_amount"].sum().reset_index()

# Step 3: Identify the top revenue city
top_city = revenue_by_city.sort_values(by="total_amount", ascending=False).iloc[0]["city"]
print("Top revenue city among Gold members:", top_city)

# Step 4: Count orders in that city (Gold members only)
orders_top_city_gold = len(gold_orders[gold_orders["city"] == top_city])
orders_top_city_gold


Top revenue city among Gold members: Chennai


1337

In [47]:
# The column used to join orders.csv and users.json is 

In [48]:
final_df = orders_df.merge(users_df, on="user_id", how="left")


NameError: name 'orders_df' is not defined